# E2 — QLoRA + DEW

In [1]:
#Check environment
import os
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [2]:
#Check dataset
import os

DATA_DIR = "/kaggle/input/datasets/ynthanhl/vitext2sql-syllable-level"

print("DATA_DIR:", DATA_DIR)
print("Files:", os.listdir(DATA_DIR))

DATA_DIR: /kaggle/input/datasets/ynthanhl/vitext2sql-syllable-level
Files: ['dev.json', 'train.json', 'test.json', 'tables.json', 'test_gold.sql']


In [3]:
#Load dataset
import json

with open(f"{DATA_DIR}/train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(f"{DATA_DIR}/dev.json", "r", encoding="utf-8") as f:
    dev_data = json.load(f)

with open(f"{DATA_DIR}/test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

with open(f"{DATA_DIR}/tables.json", "r", encoding="utf-8") as f:
    tables_data = json.load(f)

print("Train:", len(train_data))
print("Dev:", len(dev_data))
print("Test:", len(test_data))
print("Tables:", len(tables_data))

print("\nTrain keys:", list(train_data[0].keys()))
print("Table keys:", list(tables_data[0].keys()))

Train: 6831
Dev: 954
Test: 1908
Tables: 166

Train keys: ['db_id', 'query', 'query_toks', 'query_toks_no_value', 'question', 'question_toks', 'sql']
Table keys: ['column_names', 'column_names_original', 'column_types', 'db_id', 'foreign_keys', 'primary_keys', 'table_names', 'table_names_original']


In [4]:
#Create project structure
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/vietnamese_text2sql")

for folder in [
    "src",
    "src/dew",
    "src/model",
    "src/evaluation",
    "src/utils",
    "configs",
    "artifacts",
    "artifacts/predictions",
    "artifacts/metrics",
    "artifacts/error_analysis",
    "checkpoints",
    "outputs"
]:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("Created:", PROJECT_DIR.exists())

PROJECT_DIR: /kaggle/working/vietnamese_text2sql
Created: True


In [5]:
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.8 MB/s eta 0:00:00:00:0100:01


In [6]:
#Check ML packages
import transformers
import peft
import accelerate
import bitsandbytes

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

Transformers: 5.0.0
PEFT: 0.19.1
Accelerate: 1.13.0
BitsAndBytes: 0.50.2


In [7]:
#Check bitsandbytes
import torch
import bitsandbytes as bnb

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("BitsAndBytes:", bnb.__version__)
print("4-bit support:", hasattr(bnb.nn, "Linear4bit"))

CUDA: True
GPU: Tesla T4
BitsAndBytes: 0.50.2
4-bit support: True


In [8]:
#Load tokenizer
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Model:", MODEL_NAME)
print("Tokenizer:", type(tokenizer).__name__)
print("Vocab size:", tokenizer.vocab_size)
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model: Qwen/Qwen2.5-Coder-3B-Instruct
Tokenizer: Qwen2Tokenizer
Vocab size: 151643
Pad token: <|endoftext|>
EOS token: <|im_end|>


In [11]:
#Build DEW
import re
from collections import defaultdict

def build_dew_dictionary(tables_data, max_ngram=8):
    dictionaries = {}

    for schema in tables_data:
        db_id = schema["db_id"]
        entries = defaultdict(list)

        for table_id, table_name in enumerate(schema["table_names"]):
            name = table_name.strip().lower()
            if name:
                entries[name].append({
                    "entity_type": "table",
                    "table_id": table_id,
                    "column_id": None,
                    "name": table_name
                })

        for column_id, (table_id, column_name) in enumerate(schema["column_names"]):
            if table_id < 0:
                continue

            name = column_name.strip().lower()
            if name:
                entries[name].append({
                    "entity_type": "column",
                    "table_id": table_id,
                    "column_id": column_id,
                    "name": column_name
                })

        ngram_entries = defaultdict(list)

        for name, candidates in entries.items():
            tokens = name.split()

            if 1 <= len(tokens) <= max_ngram:
                ngram_entries[len(tokens)].append((name, candidates))

        dictionaries[db_id] = {
            "entries": entries,
            "ngrams": ngram_entries
        }

    return dictionaries


dew_dict = build_dew_dictionary(tables_data)

print("Databases:", len(dew_dict))
print("Academic entries:", len(dew_dict["academic"]["entries"]))

Databases: 166
Academic entries: 30


In [12]:
#Fix DEW ambiguity
def run_dew(question, db_id, dew_dict, max_ngram=8):
    tokens = question.lower().split()
    candidates = []
    db_dict = dew_dict[db_id]

    for n in range(min(max_ngram, len(tokens)), 0, -1):
        for i in range(len(tokens) - n + 1):
            phrase = " ".join(tokens[i:i+n])
            entries = db_dict["entries"].get(phrase, [])

            for entry in entries:
                candidates.append({
                    "text": phrase,
                    "start": i,
                    "end": i + n,
                    "entity_type": entry["entity_type"],
                    "table_id": entry["table_id"],
                    "column_id": entry["column_id"],
                    "name": entry["name"]
                })

    candidates.sort(
        key=lambda x: (
            -(x["end"] - x["start"]),
            x["start"]
        )
    )

    selected_spans = []
    selected = []

    for match in candidates:
        span = (match["start"], match["end"])

        if span in selected_spans:
            selected.append(match)
            continue

        overlap = any(
            not (match["end"] <= start or match["start"] >= end)
            for start, end in selected_spans
        )

        if not overlap:
            selected_spans.append(span)
            selected.append(match)

    selected.sort(
        key=lambda x: (
            x["start"],
            x["end"],
            x["entity_type"],
            x["table_id"],
            x["column_id"] if x["column_id"] is not None else -1
        )
    )

    return selected


matches = run_dew(
    train_data[0]["question"],
    train_data[0]["db_id"],
    dew_dict
)

print("Question:", train_data[0]["question"])
print("Matches:", len(matches))

for match in matches:
    print(match)

Question: hiển thị trang chủ của hội nghị PVLDB .
Matches: 5
{'text': 'trang chủ', 'start': 2, 'end': 4, 'entity_type': 'column', 'table_id': 0, 'column_id': 2, 'name': 'trang chủ'}
{'text': 'trang chủ', 'start': 2, 'end': 4, 'entity_type': 'column', 'table_id': 1, 'column_id': 6, 'name': 'trang chủ'}
{'text': 'trang chủ', 'start': 2, 'end': 4, 'entity_type': 'column', 'table_id': 5, 'column_id': 14, 'name': 'trang chủ'}
{'text': 'trang chủ', 'start': 2, 'end': 4, 'entity_type': 'column', 'table_id': 11, 'column_id': 34, 'name': 'trang chủ'}
{'text': 'hội nghị', 'start': 5, 'end': 7, 'entity_type': 'table', 'table_id': 1, 'column_id': None, 'name': 'hội nghị'}


In [13]:
#Fix DEW tag casing
def apply_dew_tags(question, db_id, dew_dict):
    matches = run_dew(question, db_id, dew_dict)

    tokens = question.split()
    normalized_tokens = question.lower().split()
    span_tags = {}

    for match in matches:
        span = (match["start"], match["end"])

        if span not in span_tags:
            original_text = " ".join(tokens[match["start"]:match["end"]])

            span_tags[span] = {
                "text": original_text,
                "entity_type": match["entity_type"],
                "candidates": []
            }

        span_tags[span]["candidates"].append({
            "table_id": match["table_id"],
            "column_id": match["column_id"]
        })

    tagged_tokens = tokens.copy()

    for (start, end), info in sorted(
        span_tags.items(),
        reverse=True
    ):
        tag = "TABLE" if info["entity_type"] == "table" else "COL"
        replacement = f"[{tag}: {info['text']}]"
        tagged_tokens[start:end] = [replacement]

    return " ".join(tagged_tokens), span_tags


question = train_data[0]["question"]
db_id = train_data[0]["db_id"]

tagged_question, tag_info = apply_dew_tags(
    question,
    db_id,
    dew_dict
)

print("Original:")
print(question)

print("\nTagged:")
print(tagged_question)

Original:
hiển thị trang chủ của hội nghị PVLDB .

Tagged:
hiển thị [COL: trang chủ] của [TABLE: hội nghị] PVLDB .


In [14]:
#Build E2 data
e2_train_data = []
e2_dev_data = []

for sample in train_data:
    tagged_question, _ = apply_dew_tags(
        sample["question"],
        sample["db_id"],
        dew_dict
    )

    e2_train_data.append({
        **sample,
        "original_question": sample["question"],
        "question": tagged_question
    })

for sample in dev_data:
    tagged_question, _ = apply_dew_tags(
        sample["question"],
        sample["db_id"],
        dew_dict
    )

    e2_dev_data.append({
        **sample,
        "original_question": sample["question"],
        "question": tagged_question
    })

print("E2 Train:", len(e2_train_data))
print("E2 Dev:", len(e2_dev_data))
print("Train tagged:", e2_train_data[0]["question"])
print("Dev tagged:", e2_dev_data[0]["question"])

E2 Train: 6831
E2 Dev: 954
Train tagged: hiển thị [COL: trang chủ] của [TABLE: hội nghị] PVLDB .
Dev tagged: Có tất cả bao nhiêu [TABLE: kiến trúc sư] nữ ?


In [19]:
#Build Qwen messages
SYSTEM_PROMPT = (
    "You are a Vietnamese Text-to-SQL system. "
    "Convert the user's question into a SQL query."
)

def build_schema_text(db_id, tables_data):
    schema = next(
        item for item in tables_data
        if item["db_id"] == db_id
    )

    lines = []

    for table_id, table_name in enumerate(schema["table_names"]):
        columns = [
            column_name
            for column_table_id, column_name in schema["column_names"]
            if column_table_id == table_id
        ]

        lines.append(
            f"Table {table_id}: {table_name} | "
            f"Columns: {', '.join(columns)}"
        )

    return "\n".join(lines)

def build_qwen_messages(sample, tables_data):
    schema_text = build_schema_text(
        sample["db_id"],
        tables_data
    )

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{schema_text}\n\n"
                f"Question:\n{sample['question']}"
            )
        },
        {
            "role": "assistant",
            "content": sample["query"]
        }
    ]

print("build_qwen_messages loaded")

build_qwen_messages loaded


In [22]:
#Build training preprocessing
MAX_SEQ_LENGTH = 2048

def preprocess_sample(sample, tables_data, tokenizer, max_seq_length=2048):
    messages = build_qwen_messages(sample, tables_data)

    prompt_ids = tokenizer.apply_chat_template(
        messages[:2],
        tokenize=True,
        add_generation_prompt=True
    )["input_ids"]

    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )["input_ids"]

    if len(full_ids) > max_seq_length:
        full_ids = full_ids[:max_seq_length]

    labels = [-100] * len(full_ids)
    prompt_length = min(len(prompt_ids), len(full_ids))

    for i in range(prompt_length, len(full_ids)):
        labels[i] = full_ids[i]

    return {
        "input_ids": full_ids,
        "labels": labels,
        "attention_mask": [1] * len(full_ids)
    }

In [23]:
#Preprocess E2 datasets
from datasets import Dataset

processed_train_e2 = [
    preprocess_sample(
        sample,
        tables_data,
        tokenizer,
        MAX_SEQ_LENGTH
    )
    for sample in e2_train_data
]

processed_dev_e2 = [
    preprocess_sample(
        sample,
        tables_data,
        tokenizer,
        MAX_SEQ_LENGTH
    )
    for sample in e2_dev_data
]

train_dataset_e2 = Dataset.from_list(processed_train_e2)
dev_dataset_e2 = Dataset.from_list(processed_dev_e2)

print("E2 Train:", len(train_dataset_e2))
print("E2 Dev:", len(dev_dataset_e2))

E2 Train: 6831
E2 Dev: 954


In [27]:
#Prepare E2 model
import gc
import torch

if "model_e1" in globals():
    del model_e1

if "model_base" in globals():
    del model_base

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)
print(
    "GPU reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

GPU allocated: 1.23 GB
GPU reserved: 2.20 GB


In [20]:
#Load E2 model
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_e2 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model_e2 = prepare_model_for_kbit_training(
    model_e2
)

lora_config_e2 = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model_e2 = get_peft_model(
    model_e2,
    lora_config_e2
)

model_e2.print_trainable_parameters()

print("E2 model loaded.")
print("Device:", next(model_e2.parameters()).device)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
E2 model loaded.
Device: cuda:0


In [21]:
#Build E2 trainer
import os
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

E2_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "checkpoints",
    "e2_qlora_dew"
)

data_collator_e2 = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_e2,
    padding=True,
    label_pad_token_id=-100
)

training_args_e2 = TrainingArguments(
    output_dir=E2_OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False
)

trainer_e2 = Trainer(
    model=model_e2,
    args=training_args_e2,
    train_dataset=train_dataset_e2,
    eval_dataset=dev_dataset_e2,
    data_collator=data_collator_e2
)

print("E2 Trainer ready.")
print("Output:", E2_OUTPUT_DIR)
print("Train:", len(train_dataset_e2))
print("Dev:", len(dev_dataset_e2))

E2 Trainer ready.
Output: /kaggle/working/vietnamese_text2sql/checkpoints/e2_qlora_dew
Train: 6831
Dev: 954


In [22]:
#Train E2
train_result_e2 = trainer_e2.train()

print("E2 training completed.")
print("Train loss:", train_result_e2.training_loss)
print("Global steps:", train_result_e2.global_step)

Epoch,Training Loss,Validation Loss
1,0.051902,0.089285
2,0.022772,0.088883


E2 training completed.
Train loss: 0.06140906908996491
Global steps: 1708


In [24]:
#Save E2 final adapter
E2_FINAL_DIR = PROJECT_DIR / "checkpoints" / "e2_qlora_dew_final"

trainer_e2.save_model(str(E2_FINAL_DIR))
tokenizer.save_pretrained(str(E2_FINAL_DIR))

print("Saved:", E2_FINAL_DIR)


Saved: /kaggle/working/vietnamese_text2sql/checkpoints/e2_qlora_dew_final


In [25]:
#Check E2 files
import os

paths = [
    "/kaggle/working/vietnamese_text2sql/checkpoints/e2_qlora_dew",
    "/kaggle/working/vietnamese_text2sql/checkpoints/e2_qlora_dew_final"
]

for path in paths:
    print("\nPATH:", path)
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        for root, dirs, files in os.walk(path):
            print(root)
            for file in files:
                print("  ", file)


PATH: /kaggle/working/vietnamese_text2sql/checkpoints/e2_qlora_dew
Exists: True
/kaggle/working/vietnamese_text2sql/checkpoints/e2_qlora_dew
/kaggle/working/vietnamese_text2sql/checkpoints/e2_qlora_dew/checkpoint-854
   adapter_config.json
   scheduler.pt
   chat_template.jinja
   adapter_model.safetensors
   optimizer.pt
   rng_state.pth
   training_args.bin
   trainer_state.json
   scaler.pt
   tokenizer.json
   README.md
   tokenizer_config.json
/kaggle/working/vietnamese_text2sql/checkpoints/e2_qlora_dew/checkpoint-1708
   adapter_config.json
   scheduler.pt
   chat_template.jinja
   adapter_model.safetensors
   optimizer.pt
   rng_state.pth
   training_args.bin
   trainer_state.json
   scaler.pt
   tokenizer.json
   README.md
   tokenizer_config.json

PATH: /kaggle/working/vietnamese_text2sql/checkpoints/e2_qlora_dew_final
Exists: True
/kaggle/working/vietnamese_text2sql/checkpoints/e2_qlora_dew_final
   adapter_config.json
   chat_template.jinja
   adapter_model.safetensors
   t

In [26]:
#Zip E2 adapter
import shutil
import os

source = "/kaggle/working/vietnamese_text2sql/checkpoints/e2_qlora_dew_final"
output = "/kaggle/working/e2_qlora_dew_final"

zip_path = shutil.make_archive(
    output,
    "zip",
    source
)

print("Created:", zip_path)
print("Size:", round(os.path.getsize(zip_path) / 1024**2, 2), "MB")

Created: /kaggle/working/e2_qlora_dew_final.zip
Size: 107.75 MB


In [9]:
#Check E2 adapter
from pathlib import Path

E2_INPUT_DIR = Path(
    "/kaggle/input/datasets/ynthanhl/e2-qlora-dew-final"
)

for path in E2_INPUT_DIR.rglob("*"):
    if path.is_file():
        print(path)

/kaggle/input/datasets/ynthanhl/e2-qlora-dew-final/adapter_model.safetensors
/kaggle/input/datasets/ynthanhl/e2-qlora-dew-final/training_args.bin
/kaggle/input/datasets/ynthanhl/e2-qlora-dew-final/adapter_config.json
/kaggle/input/datasets/ynthanhl/e2-qlora-dew-final/README.md
/kaggle/input/datasets/ynthanhl/e2-qlora-dew-final/tokenizer.json
/kaggle/input/datasets/ynthanhl/e2-qlora-dew-final/tokenizer_config.json
/kaggle/input/datasets/ynthanhl/e2-qlora-dew-final/chat_template.jinja


In [28]:
#Load E2 model
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
E2_ADAPTER_DIR = "/kaggle/input/datasets/ynthanhl/e2-qlora-dew-final"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model_e2 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model_e2 = PeftModel.from_pretrained(
    base_model_e2,
    E2_ADAPTER_DIR
)

model_e2.eval()

print("E2 adapter loaded.")
print("Device:", model_e2.device)
print("Training:", model_e2.training)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

E2 adapter loaded.
Device: cuda:0
Training: False


In [32]:
#Build E2 inference
import torch

def generate_e2_sql(sample):
    schema_text = build_schema_text(
        sample["db_id"],
        tables_data
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{schema_text}\n\n"
                f"Question:\n{sample['question']}"
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {
        key: value.to(model_e2.device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model_e2.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    prompt_length = inputs["input_ids"].shape[-1]
    generated_ids = outputs[0][prompt_length:]

    sql = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return sql

In [33]:
#Test E2 inference
sample = e2_dev_data[0]

pred_sql = generate_e2_sql(sample)

print("DB:", sample["db_id"])
print("Original:", sample["original_question"])
print("Tagged:", sample["question"])
print("Gold:", sample["query"])
print("Prediction:", pred_sql)

DB: architecture
Original: Có tất cả bao nhiêu kiến trúc sư nữ ?
Tagged: Có tất cả bao nhiêu [TABLE: kiến trúc sư] nữ ?
Gold: select count ( * ) from kiến trúc sư where giới tính = "female"
Prediction: select count ( * ) from kiến trúc sư where giới tính = "female"


In [34]:
#Run full E2 Dev inference
import json
from tqdm.auto import tqdm

e2_predictions = []

for i, sample in enumerate(tqdm(e2_dev_data)):
    pred_sql = generate_e2_sql(sample)

    e2_predictions.append({
        "index": i,
        "db_id": sample["db_id"],
        "original_question": sample["original_question"],
        "tagged_question": sample["question"],
        "gold_sql": sample["query"],
        "pred_sql": pred_sql
    })

E2_PRED_PATH = (
    "/kaggle/working/vietnamese_text2sql/"
    "artifacts/e2_official_dev_predictions.json"
)

with open(E2_PRED_PATH, "w", encoding="utf-8") as f:
    json.dump(
        e2_predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Predictions:", len(e2_predictions))
print("Saved:", E2_PRED_PATH)

  0%|          | 0/954 [00:00<?, ?it/s]

Predictions: 954
Saved: /kaggle/working/vietnamese_text2sql/artifacts/e2_official_dev_predictions.json


In [35]:
#Calculate E2 EM
import re

def normalize_sql(sql):
    sql = sql.lower().strip()
    sql = re.sub(r"\s+", " ", sql)
    return sql

e2_em_count = 0

for item in e2_predictions:
    gold = normalize_sql(item["gold_sql"])
    pred = normalize_sql(item["pred_sql"])

    if gold == pred:
        e2_em_count += 1

e2_em = e2_em_count / len(e2_predictions)

print("Total:", len(e2_predictions))
print("Exact matches:", e2_em_count)
print(f"E2 EM: {e2_em * 100:.2f}%")

Total: 954
Exact matches: 497
E2 EM: 52.10%


In [37]:
#Setup SQLite
import os

DATABASE_DIR = "/kaggle/input/datasets/ynthanhl/spider-database-166/database"

vt_map = {
    schema["db_id"]: schema
    for schema in tables_data
}

print("ViText schemas:", len(vt_map))
print("Database path:", DATABASE_DIR)
print("Database exists:", os.path.exists(DATABASE_DIR))

ViText schemas: 166
Database path: /kaggle/input/datasets/ynthanhl/spider-database-166/database
Database exists: True


In [39]:
#Build index-based SQLite bridge
import sqlite3

def build_sqlite_schema(db_id):
    schema = vt_map[db_id]

    db_path = os.path.join(
        DATABASE_DIR,
        db_id,
        f"{db_id}.sqlite"
    )

    if not os.path.exists(db_path):
        db_path = os.path.join(
            DATABASE_DIR,
            db_id,
            f"{db_id}.db"
        )

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute(
        "SELECT name FROM sqlite_master "
        "WHERE type='table' "
        "ORDER BY rowid"
    )

    sqlite_tables = [
        row[0]
        for row in cursor.fetchall()
    ]

    if len(sqlite_tables) != len(schema["table_names"]):
        raise ValueError(
            f"Table count mismatch: {db_id} | "
            f"ViText={len(schema['table_names'])} | "
            f"SQLite={len(sqlite_tables)}"
        )

    result = {}

    for table_id, vitext_table in enumerate(
        schema["table_names"]
    ):
        sqlite_table = sqlite_tables[table_id]

        cursor.execute(
            f'PRAGMA table_info("{sqlite_table}")'
        )

        sqlite_columns = [
            row[1]
            for row in cursor.fetchall()
        ]

        expected_columns = [
            column_name
            for column_table_id, column_name
            in schema["column_names"]
            if column_table_id == table_id
        ]

        if len(sqlite_columns) != len(expected_columns):
            raise ValueError(
                f"Column count mismatch: {db_id} / "
                f"table_id={table_id}"
            )

        result[table_id] = {
            "vitext_table": vitext_table,
            "sqlite_table": sqlite_table,
            "columns": sqlite_columns
        }

    conn.close()

    return result


schema_test = build_sqlite_schema("architecture")

print("DB: architecture")

for table_id, info in schema_test.items():
    print(
        table_id,
        info["vitext_table"],
        "->",
        info["sqlite_table"],
        "| columns:",
        info["columns"]
    )

DB: architecture
0 kiến trúc sư -> architect | columns: ['id', 'name', 'nationality', 'gender']
1 cầu -> bridge | columns: ['architect_id', 'id', 'name', 'location', 'length_meters', 'length_feet']
2 nhà máy -> mill | columns: ['architect_id', 'id', 'location', 'name', 'type', 'built_year', 'notes']


In [40]:
#SQL resolver
import re

def resolve_sql_for_sqlite(sql, db_id):
    schema = vt_map[db_id]
    sqlite_schema = build_sqlite_schema(db_id)

    table_map = {}
    column_map = {}

    for table_id, info in sqlite_schema.items():
        vitext_table = schema["table_names"][table_id]
        sqlite_table = info["sqlite_table"]

        table_map[vitext_table.lower()] = {
            "table_id": table_id,
            "sqlite_table": sqlite_table
        }

        column_ids = [
            column_id
            for column_id, (table_id_x, _)
            in enumerate(schema["column_names"])
            if table_id_x == table_id
        ]

        for local_index, column_id in enumerate(column_ids):
            vitext_column = schema["column_names"][column_id][1]
            sqlite_column = info["columns"][local_index]

            column_map.setdefault(
                vitext_column.lower(), []
            ).append({
                "table_id": table_id,
                "sqlite_table": sqlite_table,
                "sqlite_column": sqlite_column
            })

    resolved = sql
    aliases = {}
    referenced_tables = []

    #Find tables and aliases
    for vitext_table, info in sorted(
        table_map.items(),
        key=lambda x: -len(x[0])
    ):
        pattern = re.compile(
            rf'\b(from|join)\s+{re.escape(vitext_table)}'
            rf'(?:\s+(?:as\s+)?([A-Za-z_]\w*))?',
            flags=re.IGNORECASE
        )

        def table_callback(match):
            alias = match.group(2)

            if alias:
                aliases[alias.lower()] = info["table_id"]

            if info["table_id"] not in referenced_tables:
                referenced_tables.append(info["table_id"])

            return (
                match.group(1)
                + " "
                + f"__TABLE_{info['table_id']}__"
                + (f" {alias}" if alias else "")
            )

        resolved = pattern.sub(table_callback, resolved)

    #Replace qualified columns
    for alias, table_id in aliases.items():
        sqlite_table = sqlite_schema[table_id]["sqlite_table"]

        candidates = []

        for column_name, items in column_map.items():
            for item in items:
                if item["table_id"] == table_id:
                    candidates.append(
                        (column_name, item["sqlite_column"])
                    )

        for column_name, sqlite_column in sorted(
            candidates,
            key=lambda x: -len(x[0])
        ):
            resolved = re.sub(
                rf'\b{re.escape(alias)}\.{re.escape(column_name)}\b',
                f'{alias}.{sqlite_column}',
                resolved,
                flags=re.IGNORECASE
            )

    #Replace unqualified columns
    for column_name, items in sorted(
        column_map.items(),
        key=lambda x: -len(x[0])
    ):
        matching = [
            item for item in items
            if item["table_id"] in referenced_tables
        ]

        if len(matching) == 1:
            sqlite_column = matching[0]["sqlite_column"]

            resolved = re.sub(
                rf'(?<![\w.]){re.escape(column_name)}(?!\w)',
                sqlite_column,
                resolved,
                flags=re.IGNORECASE
            )

    #Replace remaining table placeholders
    for table_id, info in sqlite_schema.items():
        resolved = resolved.replace(
            f"__TABLE_{table_id}__",
            info["sqlite_table"]
        )

    return resolved

In [41]:
#Save baseline resolver
resolve_sql_for_sqlite_base = resolve_sql_for_sqlite

print(resolve_sql_for_sqlite_base.__name__)

resolve_sql_for_sqlite


In [42]:
#Test E2 resolver
sample = e2_predictions[0]

pred_resolved = resolve_sql_for_sqlite_base(
    sample["pred_sql"],
    sample["db_id"]
)

gold_resolved = resolve_sql_for_sqlite_base(
    sample["gold_sql"],
    sample["db_id"]
)

print("DB:", sample["db_id"])
print("Prediction:", sample["pred_sql"])
print("Resolved prediction:", pred_resolved)
print("Gold:", sample["gold_sql"])
print("Resolved gold:", gold_resolved)

DB: architecture
Prediction: select count ( * ) from kiến trúc sư where giới tính = "female"
Resolved prediction: select count ( * ) from architect where gender = "female"
Gold: select count ( * ) from kiến trúc sư where giới tính = "female"
Resolved gold: select count ( * ) from architect where gender = "female"


In [43]:
#Evaluate E2 execution
import sqlite3

total = len(e2_predictions)
pred_executable = 0
gold_executable = 0
execution_matches = 0

for item in e2_predictions:
    db_id = item["db_id"]

    db_path = os.path.join(
        DATABASE_DIR,
        db_id,
        f"{db_id}.sqlite"
    )

    if not os.path.exists(db_path):
        db_path = os.path.join(
            DATABASE_DIR,
            db_id,
            f"{db_id}.db"
        )

    pred_sql = resolve_sql_for_sqlite_base(
        item["pred_sql"],
        db_id
    )

    gold_sql = resolve_sql_for_sqlite_base(
        item["gold_sql"],
        db_id
    )

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    pred_result = None
    gold_result = None

    try:
        cursor.execute(pred_sql)
        pred_result = cursor.fetchall()
        pred_executable += 1
    except Exception:
        pass

    try:
        cursor.execute(gold_sql)
        gold_result = cursor.fetchall()
        gold_executable += 1
    except Exception:
        pass

    if (
        pred_result is not None
        and gold_result is not None
        and pred_result == gold_result
    ):
        execution_matches += 1

    conn.close()

sql_validity = pred_executable / total
execution_accuracy = execution_matches / total

print("Total:", total)
print("Prediction executable:", f"{pred_executable}/{total}")
print("Gold executable:", f"{gold_executable}/{total}")
print("Execution matches:", f"{execution_matches}/{total}")
print(f"SQL Validity: {sql_validity * 100:.2f}%")
print(f"EA: {execution_accuracy * 100:.2f}%")

Total: 954
Prediction executable: 856/954
Gold executable: 910/954
Execution matches: 616/954
SQL Validity: 89.73%
EA: 64.57%


In [45]:
#Save E2 metrics
e2_metrics = {
    "total": 954,
    "exact_matches": 497,
    "em": 52.10,
    "prediction_executable": 856,
    "gold_executable": 910,
    "execution_matches": 616,
    "sql_validity": 89.73,
    "execution_accuracy": 64.57
}

E2_METRICS_PATH = (
    "/kaggle/working/vietnamese_text2sql/"
    "artifacts/e2_official_metrics.json"
)

with open(E2_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        e2_metrics,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", E2_METRICS_PATH)

Saved: /kaggle/working/vietnamese_text2sql/artifacts/e2_official_metrics.json


In [46]:
#Check E3 setup
print("model_e2:", "model_e2" in globals())
print("tokenizer:", "tokenizer" in globals())
print("e2_dev_data:", "e2_dev_data" in globals())
print("tables_data:", "tables_data" in globals())
print("DEW:", "dew_dict" in globals())
print("Resolver:", "resolve_sql_for_sqlite_base" in globals())

model_e2: True
tokenizer: True
e2_dev_data: True
tables_data: True
DEW: True
Resolver: True


In [47]:
#Build E3 schema constraints
def build_schema_constraints(db_id):
    schema = vt_map[db_id]

    tables = [
        table_name.lower()
        for table_name in schema["table_names"]
    ]

    columns = [
        column_name.lower()
        for table_id, column_name in schema["column_names"]
        if table_id >= 0
    ]

    return {
        "tables": set(tables),
        "columns": set(columns)
    }

constraints = build_schema_constraints("architecture")

print("Tables:", constraints["tables"])
print("Columns:", constraints["columns"])

Tables: {'cầu', 'kiến trúc sư', 'nhà máy'}
Columns: {'địa điểm', 'id kiến trúc sư', 'giới tính', 'loại', 'quốc tịch', 'id', 'năm xây dựng', 'ghi chú', 'chiều dài theo feet', 'tên', 'chiều dài theo mét'}


In [49]:
#Build SQL vocabulary
SQL_KEYWORDS = {
    "select", "from", "where", "join", "on",
    "inner", "left", "right", "outer",
    "group", "by", "order", "having",
    "limit", "offset", "distinct",
    "union", "intersect", "except",
    "and", "or", "not", "in", "like",
    "is", "null", "between", "as",
    "asc", "desc",
    "count", "sum", "avg", "min", "max"
}

SQL_OPERATORS = {
    "=", "!=", "<>", "<", ">",
    "<=", ">=", "+", "-", "*", "/"
}

print("SQL keywords:", len(SQL_KEYWORDS))
print("SQL operators:", len(SQL_OPERATORS))

SQL keywords: 35
SQL operators: 11


In [50]:
#Build E3 token constraints
def build_e3_token_constraints(db_id):
    schema_constraints = build_schema_constraints(db_id)

    valid_phrases = set()

    valid_phrases.update(SQL_KEYWORDS)
    valid_phrases.update(SQL_OPERATORS)
    valid_phrases.update(schema_constraints["tables"])
    valid_phrases.update(schema_constraints["columns"])

    token_sequences = {}

    for phrase in valid_phrases:
        token_ids = tokenizer.encode(
            phrase,
            add_special_tokens=False
        )

        token_sequences[phrase] = token_ids

    return {
        "phrases": valid_phrases,
        "token_sequences": token_sequences
    }


e3_test_constraints = build_e3_token_constraints(
    "architecture"
)

print("Valid phrases:", len(e3_test_constraints["phrases"]))

for phrase in [
    "select",
    "from",
    "kiến trúc sư",
    "giới tính"
]:
    print(
        phrase,
        "->",
        e3_test_constraints["token_sequences"][phrase]
    )

Valid phrases: 60
select -> [1742]
from -> [1499]
kiến trúc sư -> [6642, 70248, 134383, 129773]
giới tính -> [8212, 51947, 128439]


In [51]:
#Build schema token variants
def build_schema_token_variants(db_id):
    constraints = build_schema_constraints(db_id)

    schema_phrases = (
        constraints["tables"]
        | constraints["columns"]
    )

    variants = {}

    for phrase in schema_phrases:
        variants[phrase] = {
            "plain": tokenizer.encode(
                phrase,
                add_special_tokens=False
            ),
            "space": tokenizer.encode(
                " " + phrase,
                add_special_tokens=False
            )
        }

    return variants


e3_schema_variants = build_schema_token_variants(
    "architecture"
)

for phrase in [
    "kiến trúc sư",
    "giới tính"
]:
    print(phrase)
    print(" plain:", e3_schema_variants[phrase]["plain"])
    print(" space:", e3_schema_variants[phrase]["space"])

kiến trúc sư
 plain: [6642, 70248, 134383, 129773]
 space: [128935, 134383, 129773]
giới tính
 plain: [8212, 51947, 128439]
 space: [128449, 128439]


In [52]:
#Build E3 logits processor
import torch
from transformers import LogitsProcessor

class SchemaTableLogitsProcessor(LogitsProcessor):
    def __init__(
        self,
        tokenizer,
        db_id,
        prompt_length
    ):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

        constraints = build_schema_constraints(db_id)

        self.table_variants = []

        for table_name in constraints["tables"]:
            token_ids = tokenizer.encode(
                " " + table_name,
                add_special_tokens=False
            )

            self.table_variants.append(token_ids)

    def __call__(self, input_ids, scores):
        generated_ids = input_ids[
            0,
            self.prompt_length:
        ].tolist()

        generated_text = self.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).lower()

        words = generated_text.rstrip().split()

        if not words:
            return scores

        #Activate after FROM or JOIN
        if words[-1] not in {"from", "join"}:
            return scores

        allowed_tokens = {
            sequence[0]
            for sequence in self.table_variants
            if sequence
        }

        if not allowed_tokens:
            return scores

        mask = torch.full_like(
            scores,
            float("-inf")
        )

        for token_id in allowed_tokens:
            mask[:, token_id] = scores[:, token_id]

        return mask

In [53]:
#Build E3 logits processor
import torch
from transformers import LogitsProcessor

class SchemaTableLogitsProcessor(LogitsProcessor):
    def __init__(
        self,
        tokenizer,
        db_id,
        prompt_length
    ):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

        constraints = build_schema_constraints(db_id)

        self.table_names = sorted(
            constraints["tables"],
            key=len,
            reverse=True
        )

    def __call__(self, input_ids, scores):
        generated_ids = input_ids[
            0,
            self.prompt_length:
        ].tolist()

        generated_text = self.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        )

        text_lower = generated_text.lower()

        #Find latest FROM or JOIN
        matches = list(
            re.finditer(
                r"\b(from|join)\s+([^\n,;]*)$",
                text_lower,
                flags=re.IGNORECASE
            )
        )

        if not matches:
            return scores

        fragment = matches[-1].group(2)

        #Stop constraint after table name is complete
        for table_name in self.table_names:
            if fragment == table_name:
                return scores

            if fragment.startswith(table_name + " "):
                return scores

        #Find valid table continuations
        candidates = [
            table_name
            for table_name in self.table_names
            if table_name.startswith(fragment)
        ]

        if not candidates:
            return scores

        allowed_tokens = set()

        for table_name in candidates:
            remaining = table_name[len(fragment):]

            if not remaining:
                continue

            token_ids = self.tokenizer.encode(
                remaining,
                add_special_tokens=False
            )

            if token_ids:
                allowed_tokens.add(token_ids[0])

        if not allowed_tokens:
            return scores

        mask = torch.full_like(
            scores,
            float("-inf")
        )

        for token_id in allowed_tokens:
            mask[:, token_id] = scores[:, token_id]

        return mask

In [54]:
#Build E3 inference
import torch
from transformers import LogitsProcessorList

def generate_e3_sql(sample):
    schema_text = build_schema_text(
        sample["db_id"],
        tables_data
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{schema_text}\n\n"
                f"Question:\n{sample['question']}"
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {
        key: value.to(model_e2.device)
        for key, value in inputs.items()
    }

    prompt_length = inputs["input_ids"].shape[-1]

    processor = SchemaTableLogitsProcessor(
        tokenizer=tokenizer,
        db_id=sample["db_id"],
        prompt_length=prompt_length
    )

    logits_processors = LogitsProcessorList([
        processor
    ])

    with torch.inference_mode():
        outputs = model_e2.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            logits_processor=logits_processors,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][prompt_length:]

    sql = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return sql

In [55]:
#Test E3 inference
sample = e2_dev_data[0]

e2_sql = generate_e2_sql(sample)
e3_sql = generate_e3_sql(sample)

print("DB:", sample["db_id"])
print("Question:", sample["question"])
print("Gold:", sample["query"])
print("E2:", e2_sql)
print("E3:", e3_sql)

DB: architecture
Question: Có tất cả bao nhiêu [TABLE: kiến trúc sư] nữ ?
Gold: select count ( * ) from kiến trúc sư where giới tính = "female"
E2: select count ( * ) from kiến trúc sư where giới tính = "female"
E3: select count ( * ) from kiến trúc sư where giới tính = "female"


In [56]:
#Test E3 JOIN
join_index = None

for i, item in enumerate(e2_predictions):
    if " join " in item["pred_sql"].lower():
        join_index = i
        break

print("Index:", join_index)

sample = e2_dev_data[join_index]

e2_sql = generate_e2_sql(sample)
e3_sql = generate_e3_sql(sample)

print("DB:", sample["db_id"])
print("Question:", sample["question"])
print("Gold:", sample["query"])
print("E2:", e2_sql)
print("E3:", e3_sql)

Index: 2
DB: architecture
Question: Cho biết [COL: tên] của người [TABLE: kiến trúc sư] đã tạo ra cây [TABLE: cầu] dài nhất cũng như là chiều dài tính bằng mét của cây [TABLE: cầu] này .
Gold: select max ( t1.chiều dài theo mét ) , t2.tên from cầu as t1 join kiến trúc sư as t2 on t1.id kiến trúc sư = t2.id
E2: select t1.tên , t2.chiều dài theo mét from kiến trúc sư as t1 join cầu as t2 on t1.id = t2.id kiến trúc sư where t2.chiều dài theo mét = ( select max ( chiều dài theo mét ) from cầu )
E3: select t1.tên , t2.chiều dài theo mét from kiến trúc sư as t1 join cầu as t2 on t1.id = t2.id kiến trúc sư where t2.chiều dài theo mét = ( select max ( chiều dài theo mét ) from cầu )


In [57]:
#Run full E3 Dev inference
import os
import json
from tqdm.auto import tqdm

E3_PRED_PATH = (
    "/kaggle/working/vietnamese_text2sql/"
    "artifacts/e3_official_dev_predictions.json"
)

e3_predictions = []

for i, sample in enumerate(tqdm(e2_dev_data)):
    pred_sql = generate_e3_sql(sample)

    e3_predictions.append({
        "index": i,
        "db_id": sample["db_id"],
        "original_question": sample["original_question"],
        "tagged_question": sample["question"],
        "gold_sql": sample["query"],
        "pred_sql": pred_sql
    })

    #Save progress
    if (i + 1) % 50 == 0:
        with open(
            E3_PRED_PATH,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                e3_predictions,
                f,
                ensure_ascii=False,
                indent=2
            )

#Save final
with open(
    E3_PRED_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        e3_predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Predictions:", len(e3_predictions))
print("Saved:", E3_PRED_PATH)

  0%|          | 0/954 [00:00<?, ?it/s]

Predictions: 954
Saved: /kaggle/working/vietnamese_text2sql/artifacts/e3_official_dev_predictions.json


In [58]:
#Calculate E3 EM
e3_em_count = 0

for item in e3_predictions:
    gold = normalize_sql(item["gold_sql"])
    pred = normalize_sql(item["pred_sql"])

    if gold == pred:
        e3_em_count += 1

e3_em = e3_em_count / len(e3_predictions)

print("Total:", len(e3_predictions))
print("Exact matches:", e3_em_count)
print(f"E3 EM: {e3_em * 100:.2f}%")

Total: 954
Exact matches: 497
E3 EM: 52.10%


In [59]:
#Evaluate E3 execution
import sqlite3

total = len(e3_predictions)
pred_executable = 0
gold_executable = 0
execution_matches = 0

for item in e3_predictions:
    db_id = item["db_id"]

    db_path = os.path.join(
        DATABASE_DIR,
        db_id,
        f"{db_id}.sqlite"
    )

    if not os.path.exists(db_path):
        db_path = os.path.join(
            DATABASE_DIR,
            db_id,
            f"{db_id}.db"
        )

    pred_sql = resolve_sql_for_sqlite_base(
        item["pred_sql"],
        db_id
    )

    gold_sql = resolve_sql_for_sqlite_base(
        item["gold_sql"],
        db_id
    )

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    pred_result = None
    gold_result = None

    try:
        cursor.execute(pred_sql)
        pred_result = cursor.fetchall()
        pred_executable += 1
    except Exception:
        pass

    try:
        cursor.execute(gold_sql)
        gold_result = cursor.fetchall()
        gold_executable += 1
    except Exception:
        pass

    if (
        pred_result is not None
        and gold_result is not None
        and pred_result == gold_result
    ):
        execution_matches += 1

    conn.close()

sql_validity = pred_executable / total
execution_accuracy = execution_matches / total

print("Total:", total)
print("Prediction executable:", f"{pred_executable}/{total}")
print("Gold executable:", f"{gold_executable}/{total}")
print("Execution matches:", f"{execution_matches}/{total}")
print(f"SQL Validity: {sql_validity * 100:.2f}%")
print(f"EA: {execution_accuracy * 100:.2f}%")

Total: 954
Prediction executable: 856/954
Gold executable: 910/954
Execution matches: 616/954
SQL Validity: 89.73%
EA: 64.57%


In [60]:
#Compare E2 and E3
different = []

for i, (e2, e3) in enumerate(
    zip(e2_predictions, e3_predictions)
):
    if e2["pred_sql"] != e3["pred_sql"]:
        different.append({
            "index": i,
            "db_id": e2["db_id"],
            "e2": e2["pred_sql"],
            "e3": e3["pred_sql"]
        })

print("Total:", len(e2_predictions))
print("Different predictions:", len(different))

for item in different[:5]:
    print("\nIndex:", item["index"])
    print("DB:", item["db_id"])
    print("E2:", item["e2"])
    print("E3:", item["e3"])

Total: 954
Different predictions: 0


In [61]:
#Find E2 schema errors
schema_error_samples = []

for i, item in enumerate(e2_predictions):
    try:
        resolved = resolve_sql_for_sqlite_base(
            item["pred_sql"],
            item["db_id"]
        )

        db_path = os.path.join(
            DATABASE_DIR,
            item["db_id"],
            f"{item['db_id']}.sqlite"
        )

        if not os.path.exists(db_path):
            db_path = os.path.join(
                DATABASE_DIR,
                item["db_id"],
                f"{item['db_id']}.db"
            )

        conn = sqlite3.connect(db_path)

        try:
            conn.execute(resolved).fetchall()
        except Exception as e:
            error = str(e)

            if (
                "no such table" in error.lower()
                or "no such column" in error.lower()
            ):
                schema_error_samples.append({
                    "index": i,
                    "db_id": item["db_id"],
                    "sql": item["pred_sql"],
                    "resolved": resolved,
                    "error": error
                })

        conn.close()

    except Exception:
        pass

print("Schema errors:", len(schema_error_samples))

for item in schema_error_samples[:10]:
    print("\nIndex:", item["index"])
    print("DB:", item["db_id"])
    print("SQL:", item["sql"])
    print("Resolved:", item["resolved"])
    print("Error:", item["error"])

Schema errors: 11

Index: 100
DB: college_3
SQL: select t1.phòng from khoá học as t1 join khoa as t2 on t1.mã số khoa = t2.mã số khoa where t1.tên khoá học = "Introduction to Computer Science"
Resolved: select t1.phòng from Course t1 join Department t2 on t1.DNO = t2.DNO where t1.CName = "Introduction to Computer Science"
Error: no such column: t1.phòng

Index: 102
DB: college_3
SQL: select t1.tên , t1.họ , t3.điểm from sinh viên as t1 join đăng ký khoá học as t2 on t1.id sinh viên = t2.id sinh viên join chuyển đổi điểm as t3 on t2.điểm = t3.bảng điểm
Resolved: select t1.Fname , t1.LName , t3.điểm from Student t1 join Enrolled_in t2 on t1.StuID = t2.StuID join Gradeconversion t3 on t2.Grade = t3.lettergrade
Error: no such column: t3.điểm

Index: 437
DB: flight_company
SQL: select t1.tên from phi công as t1 join chuyến bay as t2 on t1.id = t2.phi công join sân bay as t3 on t3.id = t2.id sân bay where t3.quốc gia = "United States" or t3.tên = "Billund Airport"
Resolved: select t1.tên fro

In [62]:
#Analyze E2 SQL errors
from collections import Counter

error_counts = Counter()
error_examples = {}

for item in e2_predictions:
    db_id = item["db_id"]

    try:
        resolved = resolve_sql_for_sqlite_base(
            item["pred_sql"],
            db_id
        )

        db_path = os.path.join(
            DATABASE_DIR,
            db_id,
            f"{db_id}.sqlite"
        )

        if not os.path.exists(db_path):
            db_path = os.path.join(
                DATABASE_DIR,
                db_id,
                f"{db_id}.db"
            )

        conn = sqlite3.connect(db_path)

        try:
            conn.execute(resolved).fetchall()

        except Exception as e:
            error = str(e)

            if "no such column" in error.lower():
                category = "no such column"

            elif "no such table" in error.lower():
                category = "no such table"

            elif "syntax error" in error.lower():
                category = "syntax error"

            elif "ambiguous column" in error.lower():
                category = "ambiguous column"

            elif "misuse" in error.lower():
                category = "misuse"

            else:
                category = "other"

            error_counts[category] += 1

            if category not in error_examples:
                error_examples[category] = {
                    "sql": item["pred_sql"],
                    "resolved": resolved,
                    "error": error
                }

        finally:
            conn.close()

    except Exception as e:
        error_counts["resolver error"] += 1

print("Total non-executable:", sum(error_counts.values()))

for category, count in error_counts.most_common():
    print(f"{category}: {count}")

print("\nExamples:")

for category, item in error_examples.items():
    print("\n", category)
    print("SQL:", item["sql"])
    print("Resolved:", item["resolved"])
    print("Error:", item["error"])

Total non-executable: 98
syntax error: 84
no such column: 9
no such table: 2
other: 2
misuse: 1

Examples:

 syntax error
SQL: select t2.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t2.địa điểm having count ( * ) >= 2
Resolved: select t2.địa điểm from cinema t1 join schedule t2 on t1.Cinema_ID = t2.Cinema_ID where t1.Capacity > 300 group by t2.địa điểm having count ( * ) >= 2
Error: near "điểm": syntax error

 no such column
SQL: select t1.phòng from khoá học as t1 join khoa as t2 on t1.mã số khoa = t2.mã số khoa where t1.tên khoá học = "Introduction to Computer Science"
Resolved: select t1.phòng from Course t1 join Department t2 on t1.DNO = t2.DNO where t1.CName = "Introduction to Computer Science"
Error: no such column: t1.phòng

 misuse
SQL: select distinct cấp bậc from thành phố order by count ( * ) asc
Resolved: select distinct Status from city order by count ( * ) asc
Error: misuse of

In [63]:
#Analyze unresolved identifiers
import re

syntax_unresolved = []

for item in e2_predictions:
    db_id = item["db_id"]

    resolved = resolve_sql_for_sqlite_base(
        item["pred_sql"],
        db_id
    )

    db_path = os.path.join(
        DATABASE_DIR,
        db_id,
        f"{db_id}.sqlite"
    )

    if not os.path.exists(db_path):
        db_path = os.path.join(
            DATABASE_DIR,
            db_id,
            f"{db_id}.db"
        )

    conn = sqlite3.connect(db_path)

    try:
        conn.execute(resolved).fetchall()

    except Exception as e:
        error = str(e)

        if "syntax error" in error.lower():
            #Vietnamese characters remaining after resolver
            has_vietnamese = bool(
                re.search(
                    r"[À-ỹĐđ]",
                    resolved
                )
            )

            if has_vietnamese:
                syntax_unresolved.append({
                    "index": item["index"],
                    "db_id": db_id,
                    "original": item["pred_sql"],
                    "resolved": resolved,
                    "error": error
                })

    finally:
        conn.close()

print("Syntax errors:", 84)
print(
    "Syntax errors with unresolved Vietnamese:",
    len(syntax_unresolved)
)

for item in syntax_unresolved[:5]:
    print("\nIndex:", item["index"])
    print("DB:", item["db_id"])
    print("Original:", item["original"])
    print("Resolved:", item["resolved"])
    print("Error:", item["error"])

Syntax errors: 84
Syntax errors with unresolved Vietnamese: 72

Index: 31
DB: cinema
Original: select t2.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t2.địa điểm having count ( * ) >= 2
Resolved: select t2.địa điểm from cinema t1 join schedule t2 on t1.Cinema_ID = t2.Cinema_ID where t1.Capacity > 300 group by t2.địa điểm having count ( * ) >= 2
Error: near "điểm": syntax error

Index: 119
DB: college_3
Original: select tên from sinh viên where id sinh viên not in ( select id sinh viên from đăng ký khoá học )
Resolved: select Fname from Student where id sinh viên not in ( select id sinh viên from Enrolled_in )
Error: near "sinh": syntax error

Index: 120
DB: college_3
Original: select tên from sinh viên where id sinh viên not in ( select id sinh viên from đăng ký khoá học )
Resolved: select Fname from Student where id sinh viên not in ( select id sinh viên from Enrolled_in )
Error: near "sin

In [65]:
#Build table-column map
def build_table_column_map(db_id):
    schema = vt_map[db_id]

    table_column_map = {}

    for table_id, table_name in enumerate(
        schema["table_names"]
    ):
        columns = [
            column_name.lower()
            for column_table_id, column_name
            in schema["column_names"]
            if column_table_id == table_id
        ]

        table_column_map[
            table_name.lower()
        ] = columns

    return table_column_map


test_map = build_table_column_map("cinema")

for table_name, columns in test_map.items():
    print(table_name, "->", columns)

phim -> ['id phim', 'xếp hạng trong sê-ri', 'mùa', 'tiêu đề', 'được đạo diễn bởi', 'ngày phát sóng ban đầu', 'mã sản xuất']
rạp chiếu phim -> ['id rạp chiếu phim', 'tên', 'năm mở cửa', 'sức chứa', 'địa điểm']
lịch chiếu phim -> ['id rạp chiếu phim', 'id phim', 'ngày', 'số lượng suất chiếu mỗi ngày', 'giá']


In [66]:
#Build alias context
def extract_alias_context(sql_text, db_id):
    table_column_map = build_table_column_map(db_id)

    alias_map = {}

    pattern = re.compile(
        r"\b(?:from|join)\s+(.+?)"
        r"(?:\s+(?:as\s+)?([A-Za-z_]\w*))"
        r"(?=\s+(?:join|on|where|group|order|having|limit|$))",
        flags=re.IGNORECASE
    )

    for match in pattern.finditer(sql_text):
        table_name = match.group(1).strip().lower()
        alias = match.group(2).lower()

        if table_name in table_column_map:
            alias_map[alias] = {
                "table": table_name,
                "columns": table_column_map[table_name]
            }

    return alias_map

In [67]:
#Test alias context
test_sql = e2_predictions[31]["pred_sql"]

alias_context = extract_alias_context(
    test_sql,
    "cinema"
)

print("SQL:", test_sql)

for alias, info in alias_context.items():
    print(
        alias,
        "->",
        info["table"],
        "->",
        info["columns"]
    )

SQL: select t2.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t2.địa điểm having count ( * ) >= 2
t1 -> rạp chiếu phim -> ['id rạp chiếu phim', 'tên', 'năm mở cửa', 'sức chứa', 'địa điểm']
t2 -> lịch chiếu phim -> ['id rạp chiếu phim', 'id phim', 'ngày', 'số lượng suất chiếu mỗi ngày', 'giá']


In [69]:
#Find qualified-column violations
qualified_violations = []

for item in e2_predictions:
    sql = item["pred_sql"].lower()
    db_id = item["db_id"]

    alias_context = extract_alias_context(
        sql,
        db_id
    )

    schema = vt_map[db_id]

    all_columns = sorted(
        {
            column_name.lower()
            for table_id, column_name
            in schema["column_names"]
            if table_id >= 0
        },
        key=len,
        reverse=True
    )

    seen = set()

    for alias, info in alias_context.items():
        valid_columns = set(info["columns"])

        for column in all_columns:
            pattern = re.compile(
                rf"(?<!\w){re.escape(alias)}\."
                rf"{re.escape(column)}(?!\w)",
                flags=re.IGNORECASE
            )

            if pattern.search(sql):
                if column not in valid_columns:
                    key = (
                        item["index"],
                        alias,
                        column
                    )

                    if key not in seen:
                        seen.add(key)

                        qualified_violations.append({
                            "index": item["index"],
                            "db_id": db_id,
                            "alias": alias,
                            "table": info["table"],
                            "column": column,
                            "sql": item["pred_sql"]
                        })

print(
    "Qualified-column violations:",
    len(qualified_violations)
)

print(
    "Affected predictions:",
    len({
        item["index"]
        for item in qualified_violations
    })
)

for item in qualified_violations[:10]:
    print("\nIndex:", item["index"])
    print("DB:", item["db_id"])
    print(
        "Alias:",
        item["alias"],
        "->",
        item["table"]
    )
    print("Invalid column:", item["column"])
    print("SQL:", item["sql"])

Qualified-column violations: 92
Affected predictions: 81

Index: 31
DB: cinema
Alias: t2 -> lịch chiếu phim
Invalid column: địa điểm
SQL: select t2.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t2.địa điểm having count ( * ) >= 2

Index: 83
DB: college_3
Alias: t1 -> khoa
Invalid column: tên
SQL: select t1.tên khoa from khoa as t1 join chuyên ngành phụ as t2 on t1.mã số khoa = t2.mã số khoa group by t1.tên khoa order by count ( * ) desc limit 1

Index: 84
DB: college_3
Alias: t1 -> khoa
Invalid column: tên
SQL: select t1.tên khoa from khoa as t1 join chuyên ngành phụ as t2 on t1.mã số khoa = t2.mã số khoa group by t1.tên khoa order by count ( * ) desc limit 1

Index: 96
DB: college_3
Alias: t1 -> khoá học
Invalid column: tên
SQL: select t1.tên khoá học from khoá học as t1 join đăng ký khoá học as t2 on t1.id khoá học = t2.mã khoá học group by t2.mã khoá học having count ( * ) >= 5

Index: 97

In [70]:
#Find real qualified-column violations
qualified_violations = []

for item in e2_predictions:
    sql = item["pred_sql"].lower()
    db_id = item["db_id"]

    alias_context = extract_alias_context(
        sql,
        db_id
    )

    schema = vt_map[db_id]

    all_columns = sorted(
        {
            column_name.lower()
            for table_id, column_name
            in schema["column_names"]
            if table_id >= 0
        },
        key=len,
        reverse=True
    )

    seen = set()

    for alias, info in alias_context.items():
        valid_columns = set(info["columns"])

        #Find alias positions
        alias_pattern = re.compile(
            rf"(?<!\w){re.escape(alias)}\.",
            flags=re.IGNORECASE
        )

        for match in alias_pattern.finditer(sql):
            start = match.end()
            remaining = sql[start:]

            #Longest schema column first
            matched_column = None

            for column in all_columns:
                if remaining.startswith(column):
                    end = len(column)

                    if (
                        end == len(remaining)
                        or not remaining[end].isalnum()
                    ):
                        matched_column = column
                        break

            if matched_column is None:
                continue

            if matched_column not in valid_columns:
                key = (
                    item["index"],
                    alias,
                    matched_column
                )

                if key not in seen:
                    seen.add(key)

                    qualified_violations.append({
                        "index": item["index"],
                        "db_id": db_id,
                        "alias": alias,
                        "table": info["table"],
                        "column": matched_column,
                        "sql": item["pred_sql"]
                    })

print(
    "Qualified-column violations:",
    len(qualified_violations)
)

print(
    "Affected predictions:",
    len({
        item["index"]
        for item in qualified_violations
    })
)

for item in qualified_violations[:10]:
    print("\nIndex:", item["index"])
    print("DB:", item["db_id"])
    print(
        "Alias:",
        item["alias"],
        "->",
        item["table"]
    )
    print("Invalid column:", item["column"])
    print("SQL:", item["sql"])

Qualified-column violations: 30
Affected predictions: 24

Index: 31
DB: cinema
Alias: t2 -> lịch chiếu phim
Invalid column: địa điểm
SQL: select t2.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t2.địa điểm having count ( * ) >= 2

Index: 100
DB: college_3
Alias: t1 -> khoá học
Invalid column: phòng
SQL: select t1.phòng from khoá học as t1 join khoa as t2 on t1.mã số khoa = t2.mã số khoa where t1.tên khoá học = "Introduction to Computer Science"

Index: 102
DB: college_3
Alias: t3 -> chuyển đổi điểm
Invalid column: điểm
SQL: select t1.tên , t1.họ , t3.điểm from sinh viên as t1 join đăng ký khoá học as t2 on t1.id sinh viên = t2.id sinh viên join chuyển đổi điểm as t3 on t2.điểm = t3.bảng điểm

Index: 133
DB: concert_singer
Alias: t2 -> ca sĩ trong buổi hoà nhạc
Invalid column: tên bài hát
SQL: select t2.tên bài hát from ca sĩ as t1 join buổi hoà nhạc as t3 on t1.id ca sĩ = t3.id buổi hoà nhạc

In [71]:
#Build valid qualified identifiers
def build_valid_qualified_identifiers(
    generated_text,
    db_id
):
    alias_context = extract_alias_context(
        generated_text,
        db_id
    )

    identifiers = []

    for alias, info in alias_context.items():
        for column in info["columns"]:
            identifiers.append(
                f"{alias}.{column}"
            )

    return sorted(
        identifiers,
        key=len,
        reverse=True
    )

In [72]:
#Test valid identifiers
test_sql = e2_predictions[31]["pred_sql"]

valid_identifiers = build_valid_qualified_identifiers(
    test_sql,
    "cinema"
)

for identifier in valid_identifiers:
    print(identifier)

t2.số lượng suất chiếu mỗi ngày
t1.id rạp chiếu phim
t2.id rạp chiếu phim
t1.năm mở cửa
t1.sức chứa
t1.địa điểm
t2.id phim
t2.ngày
t1.tên
t2.giá


In [73]:
#Build schema-aware processor
import re
import torch
from transformers import LogitsProcessor

class SchemaAwareLogitsProcessor(LogitsProcessor):
    def __init__(
        self,
        tokenizer,
        db_id,
        prompt_length
    ):
        self.tokenizer = tokenizer
        self.db_id = db_id
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores):
        generated_ids = input_ids[
            0,
            self.prompt_length:
        ].tolist()

        generated_text = self.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        )

        valid_identifiers = (
            build_valid_qualified_identifiers(
                generated_text,
                self.db_id
            )
        )

        if not valid_identifiers:
            return scores

        text_lower = generated_text.lower()

        #Find current alias.column prefix
        match = re.search(
            r"\b([a-zA-Z_]\w*)\.([^\s,()=<>]*)$",
            text_lower
        )

        if not match:
            return scores

        alias = match.group(1)
        column_prefix = match.group(2)

        prefix = f"{alias}.{column_prefix}"

        candidates = [
            identifier
            for identifier in valid_identifiers
            if identifier.startswith(prefix)
        ]

        if not candidates:
            return scores

        allowed_tokens = set()

        for identifier in candidates:
            remaining = identifier[len(prefix):]

            if not remaining:
                return scores

            token_ids = self.tokenizer.encode(
                remaining,
                add_special_tokens=False
            )

            if token_ids:
                allowed_tokens.add(
                    token_ids[0]
                )

        if not allowed_tokens:
            return scores

        mask = torch.full_like(
            scores,
            float("-inf")
        )

        for token_id in allowed_tokens:
            mask[:, token_id] = scores[:, token_id]

        return mask

In [74]:
#Build E3 inference
import torch
from transformers import LogitsProcessorList

def generate_e3_sql(sample):
    schema_text = build_schema_text(
        sample["db_id"],
        tables_data
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{schema_text}\n\n"
                f"Question:\n{sample['question']}"
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {
        key: value.to(model_e2.device)
        for key, value in inputs.items()
    }

    prompt_length = inputs["input_ids"].shape[-1]

    processor = SchemaAwareLogitsProcessor(
        tokenizer=tokenizer,
        db_id=sample["db_id"],
        prompt_length=prompt_length
    )

    logits_processors = LogitsProcessorList([
        processor
    ])

    with torch.inference_mode():
        outputs = model_e2.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            logits_processor=logits_processors,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][prompt_length:]

    sql = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return sql

In [75]:
#Test E3 index 31
test_index = 31

sample = e2_dev_data[test_index]

e2_sql = e2_predictions[test_index]["pred_sql"]
e3_sql = generate_e3_sql(sample)

print("DB:", sample["db_id"])
print("Question:", sample["question"])
print("Gold:", sample["query"])
print("E2:", e2_sql)
print("E3:", e3_sql)
print("Changed:", e2_sql != e3_sql)

DB: cinema
Question: Những [COL: địa điểm] nào có từ 2 [TABLE: rạp chiếu phim] với [COL: sức chứa] lớn hơn 300 ?
Gold: select địa điểm from rạp chiếu phim where sức chứa > 300 group by địa điểm having count ( * ) >= 2
E2: select t2.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t2.địa điểm having count ( * ) >= 2
E3: select t2.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t2.id rạp chiếu phim having count ( * ) >= 2
Changed: True


In [77]:
#Build schema-aware processor v2
import re
import torch
from transformers import LogitsProcessor

class SchemaAwareLogitsProcessor(LogitsProcessor):
    def __init__(
        self,
        tokenizer,
        db_id,
        prompt_length
    ):
        self.tokenizer = tokenizer
        self.db_id = db_id
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores):
        generated_ids = input_ids[
            0,
            self.prompt_length:
        ].tolist()

        generated_text = self.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        )

        valid_identifiers = (
            build_valid_qualified_identifiers(
                generated_text,
                self.db_id
            )
        )

        if not valid_identifiers:
            return scores

        text_lower = generated_text.lower()

        #Find current qualified identifier
        match = re.search(
            r"\b([a-zA-Z_]\w*)\.([^,()=<>]*)$",
            text_lower
        )

        if not match:
            return scores

        alias = match.group(1)
        fragment = match.group(2)

        #Stop at SQL boundary
        boundaries = [
            " from ",
            " where ",
            " join ",
            " on ",
            " group ",
            " order ",
            " having ",
            " limit ",
            " and ",
            " or "
        ]

        if any(
            boundary in fragment
            for boundary in boundaries
        ):
            return scores

        prefix = f"{alias}.{fragment}"

        candidates = [
            identifier
            for identifier in valid_identifiers
            if identifier.startswith(prefix)
        ]

        if not candidates:
            return scores

        #Allow normal generation after full column
        if prefix in candidates:
            return scores

        allowed_tokens = set()

        for identifier in candidates:
            remaining = identifier[len(prefix):]

            token_ids = self.tokenizer.encode(
                remaining,
                add_special_tokens=False
            )

            if token_ids:
                allowed_tokens.add(
                    token_ids[0]
                )

        if not allowed_tokens:
            return scores

        mask = torch.full_like(
            scores,
            float("-inf")
        )

        for token_id in allowed_tokens:
            mask[:, token_id] = scores[:, token_id]

        return mask

In [78]:
#Test E3 index 31
test_index = 31

sample = e2_dev_data[test_index]

e2_sql = e2_predictions[test_index]["pred_sql"]
e3_sql = generate_e3_sql(sample)

print("Gold:", sample["query"])
print("E2:", e2_sql)
print("E3:", e3_sql)
print("Changed:", e2_sql != e3_sql)

Gold: select địa điểm from rạp chiếu phim where sức chứa > 300 group by địa điểm having count ( * ) >= 2
E2: select t2.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t2.địa điểm having count ( * ) >= 2
E3: select t2.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t2.id rạp chiếu phim having count ( * ) >= 2
Changed: True


In [79]:
#Generate E3 candidates
def generate_e3_candidates(
    sample,
    num_beams=5,
    num_return_sequences=5
):
    schema_text = build_schema_text(
        sample["db_id"],
        tables_data
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{schema_text}\n\n"
                f"Question:\n{sample['question']}"
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {
        key: value.to(model_e2.device)
        for key, value in inputs.items()
    }

    prompt_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model_e2.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=num_beams,
            num_return_sequences=num_return_sequences,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id
        )

    candidates = []

    for output in outputs:
        generated_ids = output[prompt_length:]

        sql = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        if sql not in candidates:
            candidates.append(sql)

    return candidates

In [80]:
#Test E3 candidates
test_index = 31
sample = e2_dev_data[test_index]

candidates = generate_e3_candidates(
    sample,
    num_beams=5,
    num_return_sequences=5
)

print("Gold:", sample["query"])
print("\nCandidates:")

for i, sql in enumerate(candidates, 1):
    print(f"\n{i}. {sql}")

Gold: select địa điểm from rạp chiếu phim where sức chứa > 300 group by địa điểm having count ( * ) >= 2

Candidates:

1. select t2.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t2.địa điểm having count ( * ) >= 2

2. select t2.địa điểm from lịch chiếu phim as t1 join rạp chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t2.sức chứa > 300 group by t2.địa điểm having count ( * ) >= 2

3. select t1.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t1.địa điểm having count ( * ) >= 2

4. select t2.địa điểm from lịch chiếu phim as t1 join rạp chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t2.sức chứa > 300 group by t2.id rạp chiếu phim having count ( * ) >= 2

5. select địa điểm from rạp chiếu phim where sức chứa > 300 group by địa điểm having count ( * ) >= 2


In [81]:
#Score schema violations
def count_schema_violations(sql, db_id):
    alias_context = extract_alias_context(sql.lower(), db_id)
    schema = vt_map[db_id]
    all_columns = sorted({c.lower() for t, c in schema["column_names"] if t >= 0}, key=len, reverse=True)
    violations = 0

    for alias, info in alias_context.items():
        valid_columns = set(info["columns"])
        for match in re.finditer(rf"(?<!\w){re.escape(alias)}\.", sql.lower()):
            remaining = sql.lower()[match.end():]
            matched = next((c for c in all_columns if remaining.startswith(c) and (len(remaining) == len(c) or not remaining[len(c)].isalnum())), None)
            if matched is not None and matched not in valid_columns:
                violations += 1

    return violations

In [82]:
#Test schema scores
for i, sql in enumerate(candidates, 1):
    score = count_schema_violations(sql, sample["db_id"])
    print(f"{i}. violations={score}")
    print(sql)

1. violations=2
select t2.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t2.địa điểm having count ( * ) >= 2
2. violations=0
select t2.địa điểm from lịch chiếu phim as t1 join rạp chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t2.sức chứa > 300 group by t2.địa điểm having count ( * ) >= 2
3. violations=0
select t1.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t1.địa điểm having count ( * ) >= 2
4. violations=0
select t2.địa điểm from lịch chiếu phim as t1 join rạp chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t2.sức chứa > 300 group by t2.id rạp chiếu phim having count ( * ) >= 2
5. violations=0
select địa điểm from rạp chiếu phim where sức chứa > 300 group by địa điểm having count ( * ) >= 2


In [83]:
#Generate E3 candidates with scores
def generate_e3_candidates(sample, num_beams=5, num_return_sequences=5):
    schema_text = build_schema_text(sample["db_id"], tables_data)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Database schema:\n{schema_text}\n\nQuestion:\n{sample['question']}"}
    ]

    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    inputs = {k: v.to(model_e2.device) for k, v in inputs.items()}
    prompt_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model_e2.generate(
            **inputs, max_new_tokens=256, do_sample=False,
            num_beams=num_beams, num_return_sequences=num_return_sequences,
            return_dict_in_generate=True, output_scores=True,
            early_stopping=True, pad_token_id=tokenizer.eos_token_id
        )

    candidates = []
    for output, score in zip(outputs.sequences, outputs.sequences_scores):
        sql = tokenizer.decode(output[prompt_length:], skip_special_tokens=True).strip()
        if sql not in [x["sql"] for x in candidates]:
            candidates.append({"sql": sql, "model_score": float(score)})

    return candidates

In [84]:
#Test E3 candidate scores
sample = e2_dev_data[31]
candidates = generate_e3_candidates(sample)

for i, item in enumerate(candidates, 1):
    violations = count_schema_violations(item["sql"], sample["db_id"])
    print(f"{i}. score={item['model_score']:.4f} | violations={violations}")
    print(item["sql"])

1. score=-0.0228 | violations=2
select t2.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t2.địa điểm having count ( * ) >= 2
2. score=-0.0290 | violations=0
select t2.địa điểm from lịch chiếu phim as t1 join rạp chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t2.sức chứa > 300 group by t2.địa điểm having count ( * ) >= 2
3. score=-0.0359 | violations=0
select t1.địa điểm from rạp chiếu phim as t1 join lịch chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t1.sức chứa > 300 group by t1.địa điểm having count ( * ) >= 2
4. score=-0.0504 | violations=0
select t2.địa điểm from lịch chiếu phim as t1 join rạp chiếu phim as t2 on t1.id rạp chiếu phim = t2.id rạp chiếu phim where t2.sức chứa > 300 group by t2.id rạp chiếu phim having count ( * ) >= 2
5. score=-0.1309 | violations=0
select địa điểm from rạp chiếu phim where sức chứa > 300 group by địa điểm havi

In [85]:
#Select E3 candidate
def select_e3_candidate(candidates, db_id):
    scored = []
    for item in candidates:
        violations = count_schema_violations(item["sql"], db_id)
        scored.append({**item, "violations": violations})

    valid = [x for x in scored if x["violations"] == 0]
    pool = valid if valid else scored
    return max(pool, key=lambda x: x["model_score"])

In [86]:
#Get violation indices
violation_indices = sorted({item["index"] for item in qualified_violations})

print("Samples:", len(violation_indices))
print(violation_indices)

Samples: 24
[31, 100, 102, 133, 175, 188, 288, 328, 355, 356, 369, 371, 398, 411, 559, 562, 598, 610, 611, 612, 615, 616, 670, 869]


In [87]:
#Test E3 on violation samples
e3_violation_results = []

for n, idx in enumerate(violation_indices, 1):
    sample = e2_dev_data[idx]
    candidates = generate_e3_candidates(sample)
    selected = select_e3_candidate(candidates, sample["db_id"])

    e3_violation_results.append({
        "index": idx,
        "db_id": sample["db_id"],
        "gold": sample["query"],
        "e2": e2_predictions[idx]["pred_sql"],
        "e3": selected["sql"],
        "score": selected["model_score"],
        "violations": selected["violations"]
    })

    print(f"{n}/24 | index={idx} | violations={selected['violations']}")

1/24 | index=31 | violations=0
2/24 | index=100 | violations=0
3/24 | index=102 | violations=0
4/24 | index=133 | violations=0
5/24 | index=175 | violations=0
6/24 | index=188 | violations=0
7/24 | index=288 | violations=0
8/24 | index=328 | violations=0
9/24 | index=355 | violations=0
10/24 | index=356 | violations=0
11/24 | index=369 | violations=0
12/24 | index=371 | violations=0
13/24 | index=398 | violations=0
14/24 | index=411 | violations=0
15/24 | index=559 | violations=0
16/24 | index=562 | violations=0
17/24 | index=598 | violations=0
18/24 | index=610 | violations=0
19/24 | index=611 | violations=0
20/24 | index=612 | violations=0
21/24 | index=615 | violations=0
22/24 | index=616 | violations=0
23/24 | index=670 | violations=0
24/24 | index=869 | violations=0


In [88]:
#Compare E2 and E3
changed = [x for x in e3_violation_results if x["e2"] != x["e3"]]
e3_exact = [x for x in e3_violation_results if normalize_sql(x["e3"]) == normalize_sql(x["gold"])]
e2_exact = [x for x in e3_violation_results if normalize_sql(x["e2"]) == normalize_sql(x["gold"])]

print("Changed:", len(changed), "/ 24")
print("E2 exact:", len(e2_exact), "/ 24")
print("E3 exact:", len(e3_exact), "/ 24")
print("Remaining violations:", sum(x["violations"] > 0 for x in e3_violation_results))

Changed: 24 / 24
E2 exact: 0 / 24
E3 exact: 6 / 24
Remaining violations: 0


In [89]:
#Get correct E2 samples
correct_indices = [
    i for i, item in enumerate(e2_predictions)
    if normalize_sql(item["pred_sql"]) == normalize_sql(e2_dev_data[i]["query"])
]

control_indices = correct_indices[:24]

print("E2 correct:", len(correct_indices))
print("Control samples:", len(control_indices))
print(control_indices)

E2 correct: 497
Control samples: 24
[0, 1, 3, 4, 5, 6, 8, 9, 12, 15, 18, 19, 20, 21, 22, 24, 25, 26, 28, 32, 33, 34, 35, 36]


In [90]:
#Test E3 on control samples
e3_control_results = []

for n, idx in enumerate(control_indices, 1):
    sample = e2_dev_data[idx]
    candidates = generate_e3_candidates(sample)
    selected = select_e3_candidate(candidates, sample["db_id"])

    e3_control_results.append({
        "index": idx,
        "gold": sample["query"],
        "e2": e2_predictions[idx]["pred_sql"],
        "e3": selected["sql"],
        "violations": selected["violations"]
    })

    print(f"{n}/24 | index={idx}")

1/24 | index=0
2/24 | index=1
3/24 | index=3
4/24 | index=4
5/24 | index=5
6/24 | index=6
7/24 | index=8
8/24 | index=9
9/24 | index=12
10/24 | index=15
11/24 | index=18
12/24 | index=19
13/24 | index=20
14/24 | index=21
15/24 | index=22
16/24 | index=24
17/24 | index=25
18/24 | index=26
19/24 | index=28
20/24 | index=32
21/24 | index=33
22/24 | index=34
23/24 | index=35
24/24 | index=36


In [91]:
#Compare control results
changed = [x for x in e3_control_results if x["e2"] != x["e3"]]
e2_exact = [x for x in e3_control_results if normalize_sql(x["e2"]) == normalize_sql(x["gold"])]
e3_exact = [x for x in e3_control_results if normalize_sql(x["e3"]) == normalize_sql(x["gold"])]

print("Changed:", len(changed), "/ 24")
print("E2 exact:", len(e2_exact), "/ 24")
print("E3 exact:", len(e3_exact), "/ 24")
print("Remaining violations:", sum(x["violations"] > 0 for x in e3_control_results))

Changed: 0 / 24
E2 exact: 24 / 24
E3 exact: 24 / 24
Remaining violations: 0


In [92]:
#Build E3 hybrid predictions
e3_selected = {x["index"]: x["e3"] for x in e3_violation_results}
e3_predictions = []

for i, item in enumerate(e2_predictions):
    pred_sql = e3_selected.get(i, item["pred_sql"])

    e3_predictions.append({
        "index": i,
        "db_id": item["db_id"],
        "question": e2_dev_data[i]["question"],
        "gold_sql": e2_dev_data[i]["query"],
        "pred_sql": pred_sql
    })

print("Total:", len(e3_predictions))
print("Changed from E2:", sum(
    e3_predictions[i]["pred_sql"] != e2_predictions[i]["pred_sql"]
    for i in range(len(e3_predictions))
))

Total: 954
Changed from E2: 24


In [93]:
#Evaluate E3 EM
e3_exact_matches = sum(
    normalize_sql(item["pred_sql"]) == normalize_sql(item["gold_sql"])
    for item in e3_predictions
)

e3_em = e3_exact_matches / len(e3_predictions) * 100

print("Total:", len(e3_predictions))
print("Exact matches:", e3_exact_matches)
print(f"E3 EM: {e3_em:.2f}%")

Total: 954
Exact matches: 503
E3 EM: 52.73%


In [94]:
#Evaluate E3 execution
e3_pred_executable = 0
e3_gold_executable = 0
e3_execution_matches = 0

for item in e3_predictions:
    db_id = item["db_id"]
    db_path = os.path.join(DATABASE_DIR, db_id, f"{db_id}.sqlite")

    if not os.path.exists(db_path):
        db_path = os.path.join(DATABASE_DIR, db_id, f"{db_id}.db")

    pred_sql = resolve_sql_for_sqlite_base(item["pred_sql"], db_id)
    gold_sql = resolve_sql_for_sqlite_base(item["gold_sql"], db_id)

    conn = sqlite3.connect(db_path)
    pred_result = None
    gold_result = None

    try:
        pred_result = conn.execute(pred_sql).fetchall()
        e3_pred_executable += 1
    except Exception:
        pass

    try:
        gold_result = conn.execute(gold_sql).fetchall()
        e3_gold_executable += 1
    except Exception:
        pass

    if pred_result is not None and gold_result is not None and pred_result == gold_result:
        e3_execution_matches += 1

    conn.close()

e3_sql_validity = e3_pred_executable / len(e3_predictions) * 100
e3_ea = e3_execution_matches / len(e3_predictions) * 100

print("Total:", len(e3_predictions))
print("Prediction executable:", f"{e3_pred_executable}/{len(e3_predictions)}")
print("Gold executable:", f"{e3_gold_executable}/{len(e3_predictions)}")
print("Execution matches:", f"{e3_execution_matches}/{len(e3_predictions)}")
print(f"SQL Validity: {e3_sql_validity:.2f}%")
print(f"EA: {e3_ea:.2f}%")

Total: 954
Prediction executable: 877/954
Gold executable: 910/954
Execution matches: 624/954
SQL Validity: 91.93%
EA: 65.41%


In [95]:
#Save E3 results
import json, os

ARTIFACT_DIR = "/kaggle/working/vietnamese_text2sql/artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

e3_metrics = {
    "total": 954,
    "exact_matches": 503,
    "em": 52.73,
    "prediction_executable": 877,
    "gold_executable": 910,
    "execution_matches": 624,
    "sql_validity": 91.93,
    "execution_accuracy": 65.41
}

with open(f"{ARTIFACT_DIR}/e3_final_dev_predictions.json", "w", encoding="utf-8") as f:
    json.dump(e3_predictions, f, ensure_ascii=False, indent=2)

with open(f"{ARTIFACT_DIR}/e3_final_metrics.json", "w", encoding="utf-8") as f:
    json.dump(e3_metrics, f, ensure_ascii=False, indent=2)

print("Saved E3 predictions and metrics.")

Saved E3 predictions and metrics.


In [97]:
#E2-E3 error analysis
def is_exact(pred, gold):
    return normalize_sql(pred) == normalize_sql(gold)

analysis = []

for i in range(len(e2_dev_data)):
    gold = e2_dev_data[i]["query"]
    e2_ok = is_exact(e2_predictions[i]["pred_sql"], gold)
    e3_ok = is_exact(e3_predictions[i]["pred_sql"], gold)

    analysis.append({
        "index": i,
        "db_id": e2_dev_data[i]["db_id"],
        "e2": e2_ok,
        "e3": e3_ok
    })

wrong_to_correct = [x for x in analysis if not x["e2"] and x["e3"]]
correct_to_wrong = [x for x in analysis if x["e2"] and not x["e3"]]

print("Total:", len(analysis))
print("E2 correct:", sum(x["e2"] for x in analysis))
print("E3 correct:", sum(x["e3"] for x in analysis))
print("\nE2 -> E3")
print("Wrong -> Correct:", len(wrong_to_correct))
print("Correct -> Wrong:", len(correct_to_wrong))

Total: 954
E2 correct: 497
E3 correct: 503

E2 -> E3
Wrong -> Correct: 6
Correct -> Wrong: 0


In [98]:
#E2-E3 execution analysis
execution_analysis = []

for idx in violation_indices:
    db_id = e2_dev_data[idx]["db_id"]
    db_path = os.path.join(DATABASE_DIR, db_id, f"{db_id}.sqlite")
    if not os.path.exists(db_path):
        db_path = os.path.join(DATABASE_DIR, db_id, f"{db_id}.db")

    e2_sql = resolve_sql_for_sqlite_base(e2_predictions[idx]["pred_sql"], db_id)
    e3_sql = resolve_sql_for_sqlite_base(e3_predictions[idx]["pred_sql"], db_id)
    gold_sql = resolve_sql_for_sqlite_base(e2_dev_data[idx]["query"], db_id)

    conn = sqlite3.connect(db_path)

    def execute(sql):
        try:
            return True, conn.execute(sql).fetchall()
        except Exception:
            return False, None

    e2_valid, e2_result = execute(e2_sql)
    e3_valid, e3_result = execute(e3_sql)
    gold_valid, gold_result = execute(gold_sql)
    conn.close()

    execution_analysis.append({
        "index": idx,
        "e2_valid": e2_valid,
        "e3_valid": e3_valid,
        "e2_match": e2_valid and gold_valid and e2_result == gold_result,
        "e3_match": e3_valid and gold_valid and e3_result == gold_result
    })

print("Changed samples:", len(execution_analysis))

print("\nSQL Validity")
print("E2 invalid -> E3 valid:", sum(not x["e2_valid"] and x["e3_valid"] for x in execution_analysis))
print("E2 valid -> E3 invalid:", sum(x["e2_valid"] and not x["e3_valid"] for x in execution_analysis))

print("\nExecution")
print("E2 wrong -> E3 correct:", sum(not x["e2_match"] and x["e3_match"] for x in execution_analysis))
print("E2 correct -> E3 wrong:", sum(x["e2_match"] and not x["e3_match"] for x in execution_analysis))

Changed samples: 24

SQL Validity
E2 invalid -> E3 valid: 21
E2 valid -> E3 invalid: 0

Execution
E2 wrong -> E3 correct: 8
E2 correct -> E3 wrong: 0


In [99]:
#Load E1 predictions
import json

E1_PRED_PATH = "/kaggle/input/datasets/ynthanhl/e1-dev-predictions/e1_dev_predictions.json"

with open(E1_PRED_PATH, "r", encoding="utf-8") as f:
    e1_predictions = json.load(f)

print("E1:", len(e1_predictions))
print("E2:", len(e2_predictions))
print("E3:", len(e3_predictions))

print("\nE1 keys:", e1_predictions[0].keys())
print("E1 sample:", e1_predictions[0])

E1: 954
E2: 954
E3: 954

E1 keys: dict_keys(['db_id', 'question', 'gold', 'prediction'])
E1 sample: {'db_id': 'architecture', 'question': 'Có tất cả bao nhiêu kiến trúc sư nữ ?', 'gold': 'select count ( * ) from kiến trúc sư where giới tính = "female"', 'prediction': 'select count ( * ) from kiến trúc sư where giới tính = "female"'}


In [104]:
# SHR evaluator
def has_schema_hallucination(sql, db_id):
    sql = sql.lower()
    tables, columns = get_schema_names(db_id)
    violations = []
    sorted_tables = sorted(tables, key=len, reverse=True)

    #Check tables
    for match in re.finditer(r"\b(?:from|join)\s+", sql, re.I):
        remaining = sql[match.end():].lstrip()
        if remaining.startswith("("):
            continue

        valid_table = next((t for t in sorted_tables if remaining.startswith(t) and (len(remaining) == len(t) or remaining[len(t)] in " \t\n,;)")), None)

        if valid_table is None:
            name_match = re.match(r"(.+?)(?=\s+(?:as\s+)?[A-Za-z_]\w*\s*(?:join|on|where|group|order|having|limit|$)|\s+(?:where|join|on|group|order|having|limit)\b|[,;)])", remaining, re.I)
            if name_match:
                violations.append(("table", name_match.group(1).strip()))

    #Check qualified columns
    alias_context = extract_alias_context(sql, db_id)
    all_columns = sorted(columns, key=len, reverse=True)

    for alias, info in alias_context.items():
        valid_columns = set(info["columns"])
        for match in re.finditer(rf"(?<!\w){re.escape(alias)}\.", sql):
            remaining = sql[match.end():]
            column = next((c for c in all_columns if remaining.startswith(c) and (len(remaining) == len(c) or remaining[len(c)] in " \t\n,)=<>+-*/")), None)
            if column is not None and column not in valid_columns:
                violations.append(("column", f"{alias}.{column}"))

    return bool(violations), violations

In [105]:
#Evaluate SHR
def evaluate_shr(predictions, pred_key):
    hallucinated = 0
    details = []

    for i, item in enumerate(predictions):
        has_error, violations = has_schema_hallucination(item[pred_key], item["db_id"])

        if has_error:
            hallucinated += 1
            details.append({"index": i, "db_id": item["db_id"], "sql": item[pred_key], "violations": violations})

    return hallucinated, hallucinated / len(predictions) * 100, details

e1_shr_count, e1_shr, e1_shr_details = evaluate_shr(e1_predictions, "prediction")
e2_shr_count, e2_shr, e2_shr_details = evaluate_shr(e2_predictions, "pred_sql")
e3_shr_count, e3_shr, e3_shr_details = evaluate_shr(e3_predictions, "pred_sql")

print(f"E1 SHR: {e1_shr_count}/954 = {e1_shr:.2f}%")
print(f"E2 SHR: {e2_shr_count}/954 = {e2_shr:.2f}%")
print(f"E3 SHR: {e3_shr_count}/954 = {e3_shr:.2f}%")

E1 SHR: 39/954 = 4.09%
E2 SHR: 26/954 = 2.73%
E3 SHR: 3/954 = 0.31%


In [107]:
#Check E3 SHR
for x in e3_shr_details:
    print("\nIndex:", x["index"], "| DB:", x["db_id"])
    print("SQL:", x["sql"])
    print("Violations:", x["violations"])


Index: 133 | DB: concert_singer
SQL: select t2.tên bài hát from ca sĩ as t1 join bài hát as t2 on t1.id ca sĩ = t2.id ca sĩ where t1.tuổi > ( select avg ( tuổi ) from ca sĩ )
Violations: [('table', 'bài hát')]

Index: 437 | DB: flight_company
SQL: select t1.tên from phi công as t1 join chuyến bay as t2 on t1.id = t2.phi công join sân bay as t3 on t3.id = t2.id sân bay where t3.quốc gia = "United States" or t3.tên = "Billund Airport"
Violations: [('table', 'phi công')]

Index: 440 | DB: flight_company
SQL: select t1.tên from phi công as t1 join chuyến bay as t2 on t1.id = t2.phi công join công ty điều hành as t3 on t3.id = t2.id công ty where t3.loại = "Freight" intersect select t1.tên from phi công as t1 join chuyến bay as t2 on t1.id = t2.phi công join công ty điều hành as t3 on t3.id = t2.id công ty where t3.loại = "Dining"
Violations: [('table', 'phi công'), ('table', 'phi công')]


In [108]:
#Check schemas
for db_id in ["concert_singer", "flight_company"]:
    schema = vt_map[db_id]
    print("\nDB:", db_id)
    print("Tables:", schema["table_names"])
    for table_id, table_name in enumerate(schema["table_names"]):
        columns = [c for t, c in schema["column_names"] if t == table_id]
        print(table_id, table_name, "->", columns)


DB: concert_singer
Tables: ['sân vận động', 'ca sĩ', 'buổi hoà nhạc', 'ca sĩ trong buổi hoà nhạc']
0 sân vận động -> ['id sân vận động', 'địa điểm', 'tên', 'sức chứa', 'cao nhất', 'thấp nhất', 'số lượng tham gia trung bình']
1 ca sĩ -> ['id ca sĩ', 'tên', 'quốc gia', 'tên bài hát', 'năm phát hành bài hát', 'tuổi', 'là nam hay nữ']
2 buổi hoà nhạc -> ['id buổi hoà nhạc', 'tên buổi hoà nhạc', 'chủ đề', 'id sân vận động', 'năm']
3 ca sĩ trong buổi hoà nhạc -> ['id buổi hoà nhạc', 'id ca sĩ']

DB: flight_company
Tables: ['sân bay', 'công ty điều hành', 'chuyến bay']
0 sân bay -> ['id', 'thành phố', 'quốc gia', 'iata', 'icao', 'tên']
1 công ty điều hành -> ['id', 'tên', 'loại', 'hoạt động chính', 'hợp thành', 'cổ phần nhóm']
2 chuyến bay -> ['id', 'số hiệu', 'ngày', 'phi công', 'vận tốc', 'độ cao', 'id sân bay', 'id công ty']


In [109]:
#Save final metrics
final_metrics = {
    "E1": {"em": 50.94, "sql_validity": 88.99, "execution_accuracy": 63.94, "shr": 4.09, "schema_hallucinations": 39},
    "E2": {"em": 52.10, "sql_validity": 89.73, "execution_accuracy": 64.57, "shr": 2.73, "schema_hallucinations": 26},
    "E3": {"em": 52.73, "sql_validity": 91.93, "execution_accuracy": 65.41, "shr": 0.31, "schema_hallucinations": 3},
    "total": 954,
    "gold_executable": 910
}

FINAL_METRICS_PATH = "/kaggle/working/vietnamese_text2sql/artifacts/final_metrics.json"
with open(FINAL_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, ensure_ascii=False, indent=2)

print(FINAL_METRICS_PATH)
print(json.dumps(final_metrics, ensure_ascii=False, indent=2))

/kaggle/working/vietnamese_text2sql/artifacts/final_metrics.json
{
  "E1": {
    "em": 50.94,
    "sql_validity": 88.99,
    "execution_accuracy": 63.94,
    "shr": 4.09,
    "schema_hallucinations": 39
  },
  "E2": {
    "em": 52.1,
    "sql_validity": 89.73,
    "execution_accuracy": 64.57,
    "shr": 2.73,
    "schema_hallucinations": 26
  },
  "E3": {
    "em": 52.73,
    "sql_validity": 91.93,
    "execution_accuracy": 65.41,
    "shr": 0.31,
    "schema_hallucinations": 3
  },
  "total": 954,
  "gold_executable": 910
}


In [110]:
#Find DEW objects
[name for name in globals() if "dew" in name.lower()]

['build_dew_dictionary', 'dew_dict', 'run_dew', 'apply_dew_tags']

In [111]:
#Save DEW dictionary
DEW_PATH = "/kaggle/working/vietnamese_text2sql/artifacts/dew_dictionary.json"

with open(DEW_PATH, "w", encoding="utf-8") as f:
    json.dump(dew_dict, f, ensure_ascii=False, indent=2)

print("Saved:", DEW_PATH)
print("Databases:", len(dew_dict))
print("File size:", os.path.getsize(DEW_PATH), "bytes")

Saved: /kaggle/working/vietnamese_text2sql/artifacts/dew_dictionary.json
Databases: 166
File size: 2181286 bytes


In [112]:
#Check final functions
names = [
    "build_schema_text", "run_dew", "apply_dew_tags",
    "extract_alias_context", "build_table_column_map",
    "count_schema_violations", "generate_e3_candidates",
    "select_e3_candidate", "generate_e2_sql"
]

for name in names:
    obj = globals().get(name)
    print(name, "->", "OK" if callable(obj) else "MISSING")

build_schema_text -> OK
run_dew -> OK
apply_dew_tags -> OK
extract_alias_context -> OK
build_table_column_map -> OK
count_schema_violations -> OK
generate_e3_candidates -> OK
select_e3_candidate -> OK
generate_e2_sql -> OK


In [113]:
#Export final pipeline
import inspect, os

functions = [
    build_schema_text, run_dew, apply_dew_tags,
    build_table_column_map, extract_alias_context,
    count_schema_violations, generate_e2_sql,
    generate_e3_candidates, select_e3_candidate
]

PIPELINE_PATH = "/kaggle/working/vietnamese_text2sql/artifacts/pipeline.py"

header = '''import re
import torch

#Final NLP pipeline exported from the evaluation notebook.
#Runtime objects such as model, tokenizer, schema and DEW dictionary
#are initialized by the web backend.
'''

with open(PIPELINE_PATH, "w", encoding="utf-8") as f:
    f.write(header + "\n")
    for func in functions:
        f.write("\n\n" + inspect.getsource(func))

print("Saved:", PIPELINE_PATH)
print("File size:", os.path.getsize(PIPELINE_PATH), "bytes")

Saved: /kaggle/working/vietnamese_text2sql/artifacts/pipeline.py
File size: 7986 bytes
